## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
import sys
sys.setrecursionlimit(1000000)


def solve():
    nums = list(map(int, sys.stdin.buffer.read().split()))
    if not nums:
        return

    n = nums[0]
    fixed_a, fixed_b = nums[1], nums[2]
    arr = nums[3:3 + n]
    answer = []

    def lowest_power_of_two(x):
        return x & -x

    block = lowest_power_of_two((fixed_a - fixed_b) % n)

    def record_add(delta):
        delta %= n
        if delta == 0:
            return
        answer.append((2, delta))
        for i in range(n):
            arr[i] = (arr[i] + delta) % n

    def record_xor(mask):
        if mask == 0:
            return
        answer.append((1, mask))
        for i in range(n):
            arr[i] ^= mask

    def record_fixed_swap():
        answer.append((0, 0))
        for i in range(n):
            if arr[i] == fixed_a:
                arr[i] = fixed_b
            elif arr[i] == fixed_b:
                arr[i] = fixed_a

    def canonical_pair(x, y):
        # 在同一个低位块里，为 (x, y) 找一对和固定交换结构相同的代表值。
        rest = (y - x - block) % n
        left_rep = 0
        right_rep = 0

        step = n // 2
        while step >= 2 * block:
            if rest >= step:
                rest -= step
                right_rep += step // 2
            else:
                left_rep += step // 2
            step //= 2

        low = x & (block - 1)
        return left_rep + n // 2 + low, right_rep + low

    base_left, _ = canonical_pair(fixed_a, fixed_b)

    def swap_opposite_parity(x, y):
        # x 和 y 位于相邻奇偶大块时，可以一次映射到 fixed_a/fixed_b 完成交换。
        mapped_x, _ = canonical_pair(x, y)
        mask = mapped_x ^ base_left

        record_add(mapped_x - x)
        record_xor(mask)
        record_add(fixed_a - base_left)
        record_fixed_swap()
        record_add(base_left - fixed_a)
        record_xor(mask)
        record_add(x - mapped_x)

    def swap_value(x, y):
        if x == y:
            return

        x_part = x // block
        y_part = y // block
        if (x_part & 1) == (y_part & 1):
            # 同奇偶块不能直接换，找一个同 residue 的桥点做三次交换。
            low = x & (block - 1)
            bridge = low + block if (x_part & 1) == 0 else low
            swap_value(x, bridge)
            swap_value(y, bridge)
            swap_value(x, bridge)
            return

        swap_opposite_parity(x, y)

    def lift_residue_order(residue, length):
        # 返回一组只作用在低 length 位上的操作：正数表示加，负数表示异或。
        if length == 1:
            return True, []

        half = length // 2
        even_half = [residue[2 * i] // 2 for i in range(half)]
        odd_half = [residue[2 * i + 1] // 2 for i in range(half)]

        ok_even, even_ops = lift_residue_order(even_half, half)
        if not ok_even:
            return False, []
        ok_odd, odd_ops = lift_residue_order(odd_half, half)
        if not ok_odd:
            return False, []

        ops = []
        if residue[0] & 1:
            ops.append(1 if length == 2 else -1)

        even_mask = 0
        for op in even_ops:
            if op > 0:
                ops.extend((-1, 1))
            else:
                op <<= 1
                ops.append(op)
                even_mask ^= -op
        if even_mask:
            ops.append(-even_mask)

        odd_mask = 0
        for op in odd_ops:
            if op > 0:
                ops.extend((1, -1))
            else:
                op <<= 1
                ops.append(op)
                odd_mask ^= -op

        if (even_mask & half) != (odd_mask & half):
            return False, []
        even_mask %= half
        odd_mask %= half
        if even_mask != odd_mask:
            return False, []

        compact = []
        for op in ops:
            if compact and op < 0 and compact[-1] < 0:
                merged = -((-compact[-1]) ^ (-op))
                if merged:
                    compact[-1] = merged
                else:
                    compact.pop()
            else:
                compact.append(op)

        return True, compact

    if block > 1:
        residue_now = [arr[i] & (block - 1) for i in range(block)]
        ok, prep_ops = lift_residue_order(residue_now, block)
        if not ok:
            print(-1)
            return
        for op in prep_ops:
            if op > 0:
                record_add(op)
            else:
                record_xor(-op)

    for r in range(block):
        if sorted(arr[r::block]) != list(range(r, n, block)):
            print(-1)
            return

        pos = r
        while pos < n:
            if arr[pos] != pos:
                swap_value(pos, arr[pos])
                if len(answer) > 32768:
                    print(-1)
                    return
            pos += block

    if arr != list(range(n)) or len(answer) > 32768:
        print(-1)
        return

    out = [str(len(answer))]
    for kind, value in answer:
        if kind == 0:
            out.append('0')
        else:
            out.append(f'{kind} {value}')
    sys.stdout.write('\n'.join(out))


if __name__ == '__main__':
    solve()

## B 长跑

In [ ]:
import sys
from array import array
import heapq


def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    pos = 0
    out = []

    while pos < len(data):
        n, length, maxn, coins = data[pos:pos + 4]
        pos += 4

        # 同一个位置可能有多家补给站，只需要保留最便宜的一家。
        stations = {}
        for _ in range(n):
            p, cost = data[pos:pos + 2]
            pos += 2
            if 0 < p < length:
                stations[p] = min(stations.get(p, cost), cost)

        # 堆里保存最近 maxn 距离内可到达的状态：(总花费, 位置)。
        # 起点体力已满，所以起点状态不需要花钱。
        heap = [(0, 0)]
        for p, cost in sorted(stations.items()):
            while heap and heap[0][1] < p - maxn:
                heapq.heappop(heap)
            if heap:
                heapq.heappush(heap, (heap[0][0] + cost, p))

        while heap and heap[0][1] < length - maxn:
            heapq.heappop(heap)

        out.append("Yes" if heap and heap[0][0] <= coins else "No")

    print("\n".join(out))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:
import sys
from array import array


BASE = 911382323
MASK = (1 << 64) - 1


def get_pal_radius(s):
    n = len(s)

    odd = array('I', [0]) * n
    left, right = 0, -1
    for center in range(n):
        if center > right:
            r = 1
        else:
            r = odd[left + right - center]
            bound = right - center + 1
            if bound < r:
                r = bound
        while center - r >= 0 and center + r < n and s[center - r] == s[center + r]:
            r += 1
        odd[center] = r
        if center + r - 1 > right:
            left = center - r + 1
            right = center + r - 1

    even = array('I', [0]) * n
    left, right = 0, -1
    for center in range(n):
        if center > right:
            r = 0
        else:
            r = even[left + right - center + 1]
            bound = right - center + 1
            if bound < r:
                r = bound
        while center - r - 1 >= 0 and center + r < n and s[center - r - 1] == s[center + r]:
            r += 1
        even[center] = r
        if center + r - 1 > right:
            left = center - r
            right = center + r - 1

    return odd, even


def main():
    data = sys.stdin.buffer.read().split()
    if not data:
        return

    n = int(data[0])
    a = data[1]
    b = data[2]

    odd_a, even_a = get_pal_radius(a)
    odd_b, even_b = get_pal_radius(b)

    # 先把只选 A 或只选 B 的答案算出来。
    ans = 1
    for r in odd_a:
        ans = max(ans, (r << 1) - 1)
    for r in even_a:
        ans = max(ans, r << 1)
    for r in odd_b:
        ans = max(ans, (r << 1) - 1)
    for r in even_b:
        ans = max(ans, r << 1)

    rev_a = a[::-1]
    power = array('Q', [1]) * (n + 1)
    hash_rev_a = array('Q', [0]) * (n + 1)
    hash_b = array('Q', [0]) * (n + 1)

    for i in range(n):
        power[i + 1] = (power[i] * BASE) & MASK
        hash_rev_a[i + 1] = (hash_rev_a[i] * BASE + rev_a[i]) & MASK
        hash_b[i + 1] = (hash_b[i] * BASE + b[i]) & MASK

    def same(rev_pos, b_pos, length):
        left_hash = (hash_rev_a[rev_pos + length] - hash_rev_a[rev_pos] * power[length]) & MASK
        right_hash = (hash_b[b_pos + length] - hash_b[b_pos] * power[length]) & MASK
        return left_hash == right_hash

    def try_candidate(a_end, b_begin, middle_len):
        nonlocal ans
        if a_end < 0 or b_begin >= n:
            return

        limit = min(a_end + 1, n - b_begin)
        if limit <= 0 or middle_len + (limit << 1) <= ans:
            return

        rev_pos = n - 1 - a_end
        need = ((ans - middle_len) >> 1) + 1
        if need > limit or rev_a[rev_pos] != b[b_begin] or not same(rev_pos, b_begin, need):
            return

        matched = need
        if matched < limit:
            step = 1
            while matched + step <= limit and same(rev_pos, b_begin, matched + step):
                matched += step
                step <<= 1

            high = min(limit, matched + step - 1)
            while matched < high:
                mid = (matched + high + 1) >> 1
                if same(rev_pos, b_begin, mid):
                    matched = mid
                else:
                    high = mid - 1

        ans = max(ans, middle_len + (matched << 1))

    # 中间回文在 A 内部。
    for center, r in enumerate(odd_a):
        try_candidate(center - r, center + r - 1, (r << 1) - 1)
    for center, r in enumerate(even_a):
        if r:
            try_candidate(center - r - 1, center + r - 1, r << 1)

    # 中间回文在 B 内部。
    for center, r in enumerate(odd_b):
        try_candidate(center - r + 1, center + r, (r << 1) - 1)
    for center, r in enumerate(even_b):
        if r:
            try_candidate(center - r, center + r, r << 1)

    # 中间为空，直接从 A/B 的同一分界处向两边扩。
    for split in range(n):
        try_candidate(split, split, 0)

    print(ans)


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
import sys
from bisect import bisect_right, insort


def main():
    data = sys.stdin.buffer.read().split()
    pos = 0
    total = len(data)
    out = []

    while pos < total:
        m = int(data[pos])
        pos += 1

        # 对每个编号，合法序列必须按 I/O 交替。
        # 若第一次已知操作是 O，则要在它之前找一个 ? 补成 I。
        # 若连续两次已知操作相同，则要在两者之间找一个 ? 补成相反操作。
        last_op = {}
        last_pos = {}
        qs = []
        ans = -1

        for line in range(1, m + 1):
            op = data[pos]
            pos += 1

            if op == b'I' or op == b'O':
                x = int(data[pos])
                pos += 1

                if ans != -1:
                    continue

                need_after = None
                if x not in last_op:
                    if op == b'O':
                        need_after = 0
                else:
                    if last_op[x] == op:
                        need_after = last_pos[x]

                if need_after is not None:
                    idx = bisect_right(qs, need_after)
                    if idx == len(qs):
                        ans = line
                    else:
                        qs.pop(idx)

                last_op[x] = op
                last_pos[x] = line

            else:
                if ans == -1:
                    insort(qs, line)

        out.append(str(ans))

    print("\n".join(out))


if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
import sys


def main():
    data = sys.stdin.buffer.read().split()
    idx = 0
    n = int(data[idx]); idx += 1

    xs = [0] * n
    ys = [0] * n
    for i in range(n):
        xs[i] = int(data[idx]); idx += 1
        ys[i] = int(data[idx]); idx += 1

    parent = list(range(n))

    def find(x):
        # 路径压缩，顺手把树压扁，后面查得更快。
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb

    # 同一行（y 相同）或同一列（x 相同）的两个点本来就能互相到达，先把它们并到一起。
    # 按 x 分组，组内任意两点同列，串成一个连通块就行。
    by_x = {}
    by_y = {}
    for i in range(n):
        if xs[i] in by_x:
            union(i, by_x[xs[i]])
        else:
            by_x[xs[i]] = i

        if ys[i] in by_y:
            union(i, by_y[ys[i]])
        else:
            by_y[ys[i]] = i

    # 统计连通块个数。
    blocks = len(set(find(i) for i in range(n)))

    # 新加一个点只有一个横坐标和一个纵坐标，最多把两个连通块连起来，
    # 所以把所有块连通至少要加 (块数 - 1) 个点。
    print(blocks - 1)


if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
import sys


def main():
    data = sys.stdin.buffer.read().splitlines()

    pattern = data[0].strip()
    n = int(data[1])

    parts = pattern.split(b'*')
    start_star = pattern.startswith(b'*')
    end_star = pattern.endswith(b'*')

    specs = []
    for part in parts:
        chunks = []
        i = 0
        while i < len(part):
            if part[i] == ord('?'):
                i += 1
            else:
                j = i
                while j < len(part) and part[j] != ord('?'):
                    j += 1
                chunks.append((i, part[i:j]))
                i = j
        specs.append((len(part), chunks))

    def build_masks(s):
        masks = [0] * 26
        for i, c in enumerate(s):
            masks[c - 97] |= 1 << i
        return masks

    def match_at(s, spec, pos):
        length, chunks = spec
        if pos < 0 or pos + length > len(s):
            return False

        for off, chunk in chunks:
            if not s.startswith(chunk, pos + off):
                return False

        return True

    def find_seg(s, masks, spec, left, right):
        length, chunks = spec

        if length == 0:
            return left

        max_start = right - length
        if left > max_start:
            return -1

        # cand 的第 i 位表示从 i 位置开始有可能匹配当前这一段。
        cand = ((1 << (max_start + 1)) - 1) ^ ((1 << left) - 1)

        # 先用字母出现位置做位运算筛选，减少需要逐个检查的位置。
        for off, chunk in chunks:
            if len(chunk) == 0:
                continue

            c = chunk[0] - 97
            cand &= masks[c] >> off

            if len(chunk) > 1:
                c = chunk[-1] - 97
                cand &= masks[c] >> (off + len(chunk) - 1)

            if len(chunk) > 2:
                mid = len(chunk) // 2
                c = chunk[mid] - 97
                cand &= masks[c] >> (off + mid)

            if cand == 0:
                return -1

        while cand:
            lowbit = cand & -cand
            pos = lowbit.bit_length() - 1

            if match_at(s, spec, pos):
                return pos

            cand ^= lowbit

        return -1

    ans = []

    for idx in range(n):
        s = data[idx + 2].strip()
        masks = build_masks(s)

        if len(specs) == 1:
            if len(s) == specs[0][0] and match_at(s, specs[0], 0):
                ans.append(b"YES")
            else:
                ans.append(b"NO")
            continue

        left = 0
        right = len(s)
        l = 0
        r = len(specs) - 1
        ok = True

        # 如果不是以 * 开头，第一段必须从文件名开头开始匹配。
        if not start_star:
            if not match_at(s, specs[0], 0):
                ok = False
            else:
                left = specs[0][0]
                l = 1

        # 如果不是以 * 结尾，最后一段必须匹配到文件名结尾。
        if ok and not end_star:
            last_len = specs[-1][0]
            pos = len(s) - last_len

            if pos < left or not match_at(s, specs[-1], pos):
                ok = False
            else:
                right = pos
                r -= 1

        # 中间的各段按顺序出现即可。
        if ok:
            for i in range(l, r + 1):
                if specs[i][0] == 0:
                    continue

                pos = find_seg(s, masks, specs[i], left, right)

                if pos == -1:
                    ok = False
                    break

                left = pos + specs[i][0]

        ans.append(b"YES" if ok else b"NO")

    sys.stdout.buffer.write(b"\n".join(ans))


if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
import sys


def main():
    n = int(sys.stdin.readline().strip())
    order = sys.stdin.readline().strip().split()

    pegs = "ABC"

    # priority 用来记录每种操作的优先级，数字越小表示越先考虑。
    priority = {}
    for i, op in enumerate(order):
        priority[op] = i

    def other(a, b):
        for p in pegs:
            if p != a and p != b:
                return p

    # cnt[k][p] 表示 k 个盘子都在 p 柱子上时，按题目规则完成移动需要的步数。
    cnt = [{p: 0 for p in pegs} for _ in range(n + 1)]

    # to[k][p] 表示 k 个盘子从 p 柱子开始移动后，最后整体会停在哪根柱子上。
    to = [{p: "" for p in pegs} for _ in range(n + 1)]

    # 只有一个盘子时，只要从当前柱子出发，选优先级最高的移动即可。
    for p in pegs:
        best = None
        for q in pegs:
            if p == q:
                continue
            op = p + q
            if best is None or priority[op] < priority[best]:
                best = op

        cnt[1][p] = 1
        to[1][p] = best[1]

    for k in range(2, n + 1):
        for start in pegs:
            # 先把上面的 k-1 个小盘子移走，最大盘子才可以移动。
            total = cnt[k - 1][start]
            small_pos = to[k - 1][start]

            # 最大盘子只能移动到当前空出来的那根柱子。
            big_pos = other(start, small_pos)
            total += 1

            while True:
                # 再移动 k-1 个小盘子，看它们是否能盖到最大盘子上。
                total += cnt[k - 1][small_pos]
                small_pos = to[k - 1][small_pos]

                if small_pos == big_pos:
                    cnt[k][start] = total
                    to[k][start] = big_pos
                    break

                # 如果还没有完成，最大盘子继续移动到剩下的空柱子。
                total += 1
                big_pos = other(big_pos, small_pos)

    print(cnt[n]["A"])


if __name__ == "__main__":
    main()

## H 马步距离

In [ ]:
import sys


def knight_distance(dx, dy):
    # 棋盘足够大，所以只需要考虑两个点横纵坐标差的绝对值。
    x = max(abs(dx), abs(dy))
    y = min(abs(dx), abs(dy))

    # 起点和终点相同，不需要移动。
    if x == 0 and y == 0:
        return 0

    # 这两个位置是马步距离公式中的特殊情况。
    if x == 1 and y == 0:
        return 3

    if x == 2 and y == 2:
        return 4

    # 一方面，马每步最多让较大坐标方向接近 2；
    # 另一方面，每步横纵坐标变化总量最多可以看作 3。
    ans = max((x + 1) // 2, (x + y + 2) // 3)

    # 马每走一步都会改变坐标和的奇偶性，所以最后要修正奇偶性。
    if (ans + x + y) % 2 == 1:
        ans += 1

    return ans


def main():
    xp, yp, xs, ys = map(int, sys.stdin.readline().split())

    dx = xs - xp
    dy = ys - yp

    print(knight_distance(dx, dy))


if __name__ == "__main__":
    main()

## I 直方图最大矩形

In [ ]:
from typing import List


class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        # 我的想法是：如果一个柱子确定为矩形的高度，
        # 那么需要知道它左右两边最远能扩展到哪里。
        # 这里用栈来记录还没有确定右边界的柱子下标。
        stack = []
        ans = 0

        # 两边补 0 是为了让边界情况更好处理，
        # 特别是最后一个 0 可以把栈里剩下的柱子都算出来。
        heights = [0] + heights + [0]

        for i, h in enumerate(heights):
            # 当前高度变小的时候，说明前面较高的柱子不能再往右延伸了。
            while stack and heights[stack[-1]] > h:
                cur = stack.pop()
                height = heights[cur]

                # 弹出后，栈顶就是左边第一个比它矮的位置，
                # 当前 i 是右边第一个比它矮的位置，所以中间部分就是它能覆盖的宽度。
                width = i - stack[-1] - 1
                ans = max(ans, height * width)

            stack.append(i)

        return ans

## J 消防局的设立

In [ ]:
import sys


def main():
    data = sys.stdin.buffer.read().split()
    pos = 0
    n = int(data[pos]); pos += 1

    parent = [0] * (n + 1)
    adj = [[] for _ in range(n + 1)]

    # 题目保证 a[i] < i，所以 1 号是根，每个点的父亲编号都比自己小。
    for i in range(2, n + 1):
        p = int(data[pos]); pos += 1
        parent[i] = p
        adj[i].append(p)
        adj[p].append(i)

    # 父亲编号都比自己小，按编号顺序就能直接把每个点的深度递推出来。
    depth = [0] * (n + 1)
    for i in range(2, n + 1):
        depth[i] = depth[parent[i]] + 1

    # 先处理最深的点，这样建站时能尽量往上放，覆盖的范围最大。
    order = sorted(range(1, n + 1), key=lambda x: depth[x], reverse=True)

    covered = [False] * (n + 1)
    ans = 0

    for u in order:
        if covered[u]:
            continue

        # 这个点还没被覆盖，就把消防局建在它往上两层的祖先处（到根就停下）。
        g = u
        for _ in range(2):
            if parent[g] != 0:
                g = parent[g]

        ans += 1

        # 从建站点出发，把距离不超过 2 的基地全部标记成已覆盖。
        covered[g] = True
        for v in adj[g]:
            covered[v] = True
            for w in adj[v]:
                covered[w] = True

    print(ans)


if __name__ == "__main__":
    main()